# Preprocessing & Sentiment Labeling — FUZZ CHANNEL Comments

**Project 2 | High Performance Data Processing (SECP3133)**

---

### What this notebook does
| Step | Input | Output |
|---|---|---|
| Load & Dedupe | `data/raw_data/youtube_comments_raw.csv` | Clean DataFrame |
| Text Cleaning | `comment_text` | `clean_text` (transformer), `clean_text_ml` (classical ML) |
| Tokenization & Normalization | `clean_text_ml` | `tokens` (stopword-removed, stemmed) |
| Sentiment Labeling | `clean_text` | `sentiment_label`, `sentiment_score` |
| Manual Verification | random 300-row sample | `data/sample_manual_check.csv` |
| Save | Final DataFrame | `data/cleaned_data.csv` |

**Model:** `cardiffnlp/twitter-xlm-roberta-base-sentiment` — multilingual XLM-RoBERTa fine-tuned on Twitter data. Handles mixed Malay + English natively.  
**Labels:** `positive` · `neutral` · `negative`

## 1. Setup & Imports

Required packages — install any that are missing:
```bash
pip install pandas nltk PySastrawi transformers torch emoji tqdm

In [2]:

import re
import html
import os
import random

import emoji
import numpy as np
import pandas as pd
import torch
import nltk
from tqdm import tqdm
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from transformers import pipeline
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

print("All libraries imported successfully.")
print(f"CUDA available: {torch.cuda.is_available()}")

All libraries imported successfully.
CUDA available: True


## 2. Load & Deduplicate

Steps:
1. Read the raw CSV with `utf-8-sig` encoding (matches the BOM written during collection).
2. Inspect shape and a sample.
3. Drop rows where `comment_text` is empty or whitespace-only.
4. Drop duplicate `comment_id` rows — keeps the first occurrence.

In [3]:
RAW_PATH = r"c:\UTM things\Year 3\SEM 2\HDPD\Project 2\data\raw_data\youtube_comments_raw.csv"

df = pd.read_csv(RAW_PATH, encoding="utf-8-sig")

print(f"Loaded  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns : {list(df.columns)}\n")

# ── Drop empty comment_text ────────────────────────────────────────────────────
empty_mask = df["comment_text"].isna() | (df["comment_text"].str.strip() == "")
print(f"Empty comment_text rows : {empty_mask.sum()}")
df = df[~empty_mask].copy()

# ── Drop duplicate comment_ids ─────────────────────────────────────────────────
dupes = df.duplicated(subset="comment_id").sum()
print(f"Duplicate comment_ids   : {dupes}")
df = df.drop_duplicates(subset="comment_id", keep="first").reset_index(drop=True)

print(f"\nClean shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

Loaded  : 12,000 rows × 10 columns
Columns : ['video_id', 'video_title', 'video_published_at', 'comment_id', 'author', 'comment_text', 'like_count', 'reply_count', 'published_at', 'updated_at']

Empty comment_text rows : 1
Duplicate comment_ids   : 0

Clean shape : 11,999 rows × 10 columns


,video_id,video_title,video_published_at,comment_id,author,comment_text,like_count,reply_count,published_at,updated_at
0,5yx4wSM9Nc0,Selamat Tinggal iPad... Dah Jumpa Tablet Lagi ...,2026-06-18T12:00:01Z,UgxawzseWkTM3H2BX8d4AaABAg,@faizulirfan22,"For me, tablet/ipad tak boleh replace the lapt...",23,0,2026-06-18T17:56:46Z,2026-06-18T17:56:46Z
1,5yx4wSM9Nc0,Selamat Tinggal iPad... Dah Jumpa Tablet Lagi ...,2026-06-18T12:00:01Z,UgzNc8pP_d0F8BDXKKR4AaABAg,@adzalanmdamin8154,tablet huawei da lama pintas ipad pun...4 tahu...,11,1,2026-06-19T05:47:30Z,2026-06-19T05:47:30Z
2,5yx4wSM9Nc0,Selamat Tinggal iPad... Dah Jumpa Tablet Lagi ...,2026-06-18T12:00:01Z,UgyLGZboSbPEVGqqTxd4AaABAg,@ayeemasrii,hopefully through update features je yg boleh ...,1,0,2026-06-18T20:05:41Z,2026-06-18T20:05:41Z


## 3. Text Cleaning

Two cleaned versions are produced from `comment_text`:

| Column | Purpose | What is stripped |
|---|---|---|
| `clean_text` | Transformer labeling + transformer model training | HTML entities, URLs, @mentions, extra whitespace. Emojis and original casing **kept** — XLM-RoBERTa was trained on tweets and handles both. |
| `clean_text_ml` | Classical ML (Naive Bayes, SVM, TF-IDF) | Everything above + emojis, hashtags, punctuation, digits, lowercased. |

The original `comment_text` column is preserved untouched.

In [4]:
def clean_light(text: str) -> str:
    """Light cleaning for transformer — keep emojis and casing."""
    text = html.unescape(text)                        # &#39; → '
    text = re.sub(r"http\S+|www\.\S+", "", text)     # remove URLs
    text = re.sub(r"@\w+", "", text)                  # remove @mentions
    text = re.sub(r"\s+", " ", text).strip()          # collapse whitespace
    return text


def clean_full(text: str) -> str:
    """Full cleaning for classical ML — strip all noise, lowercase."""
    text = clean_light(text)
    text = emoji.replace_emoji(text, replace="")      # remove emojis
    text = re.sub(r"#\w+", "", text)                  # remove hashtags
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)              # keep a-z and spaces only
    text = re.sub(r"\s+", " ", text).strip()          # collapse whitespace
    return text


print("Cleaning functions defined.")

Cleaning functions defined.


In [5]:
df["clean_text"]    = df["comment_text"].astype(str).map(clean_light)
df["clean_text_ml"] = df["comment_text"].astype(str).map(clean_full)

# ── Sanity check ───────────────────────────────────────────────────────────────
print("=== Before vs After (5 samples) ===\n")
sample_idx = df.sample(5, random_state=42).index
for i in sample_idx:
    print(f"ORIGINAL : {df.loc[i, 'comment_text'][:120]}")
    print(f"LIGHT    : {df.loc[i, 'clean_text'][:120]}")
    print(f"FULL     : {df.loc[i, 'clean_text_ml'][:120]}")
    print()

# Flag rows where light cleaning emptied the text (use original as fallback)
empty_after = df["clean_text"].str.strip() == ""
print(f"Rows empty after light cleaning : {empty_after.sum()} (will use comment_text as fallback)")
df.loc[empty_after, "clean_text"] = df.loc[empty_after, "comment_text"].map(clean_light)

=== Before vs After (5 samples) ===

ORIGINAL : Tajuk misleading, rm4000 tapi starting dah phone s26 ultra harga 5k lebih. L fuzz
LIGHT    : Tajuk misleading, rm4000 tapi starting dah phone s26 ultra harga 5k lebih. L fuzz
FULL     : tajuk misleading rm tapi starting dah phone s ultra harga k lebih l fuzz

ORIGINAL : Mcm hnpon tuan2.; 5-10 thun pakai, bateri MESTI rosak (bateri health menurun @ bateri kembung).. Time tu majority nye hn
LIGHT    : Mcm hnpon tuan2.; 5-10 thun pakai, bateri MESTI rosak (bateri health menurun @ bateri kembung).. Time tu majority nye hn
FULL     : mcm hnpon tuan thun pakai bateri mesti rosak bateri health menurun bateri kembung time tu majority nye hnpon tu trus sim

ORIGINAL : first
LIGHT    : first
FULL     : first

ORIGINAL : Honor ni jd perintis,pembuka jln tel lasak harga murah...terbaik honor👍👍👍👍👍👍
LIGHT    : Honor ni jd perintis,pembuka jln tel lasak harga murah...terbaik honor👍👍👍👍👍👍
FULL     : honor ni jd perintispembuka jln tel lasak harga murahter

## 4. Tokenization & Normalization

Produces a `tokens` column (space-joined string) suitable for TF-IDF vectorizers.

**Pipeline for each comment (`clean_text_ml`):**
1. Word tokenize with NLTK `word_tokenize`.
2. Remove combined English + Malay stopwords.
3. Remove tokens shorter than 2 characters.
4. Apply `WordNetLemmatizer` (English morphology).
5. Apply Sastrawi stemmer (Malay morphology).

For mixed-language text, running both lemmatizer → stemmer is a standard approximation: the lemmatizer handles English, and the stemmer handles Malay affixes (me-, ber-, -kan, etc.). Neither step damages words from the other language significantly.

In [6]:
# ── NLTK downloads ─────────────────────────────────────────────────────────────
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

# ── English resources ──────────────────────────────────────────────────────────
en_stopwords = set(stopwords.words("english"))
lemmatizer   = WordNetLemmatizer()

# ── Malay resources (Sastrawi) ─────────────────────────────────────────────────
malay_stemmer    = StemmerFactory().create_stemmer()
malay_stopwords  = set(StopWordRemoverFactory().get_stop_words())

# Combined stopword set
all_stopwords = en_stopwords | malay_stopwords

print(f"English stopwords : {len(en_stopwords)}")
print(f"Malay stopwords   : {len(malay_stopwords)}")
print(f"Combined          : {len(all_stopwords)}")

English stopwords : 198
Malay stopwords   : 809
Combined          : 999


In [7]:
print(df.columns.tolist())
print(df.shape)

['video_id', 'video_title', 'video_published_at', 'comment_id', 'author', 'comment_text', 'like_count', 'reply_count', 'published_at', 'updated_at', 'clean_text', 'clean_text_ml']
(11999, 12)


In [8]:
def normalize(text: str) -> str:
    """Tokenize → remove stopwords → lemmatize (EN) → stem (MY)."""
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in all_stopwords and len(t) > 1]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]   # English lemmatization
    tokens = [malay_stemmer.stem(t) for t in tokens]     # Malay stemming
    return " ".join(tokens)


print("Applying normalization (may take ~1 min for 12k rows) ...")
df["tokens"] = df["clean_text_ml"].map(normalize)

print("Done.")
print("\nSample tokens:")
for i in df.sample(5, random_state=1).index:
    print(f"  ML text : {df.loc[i, 'clean_text_ml'][:80]}")
    print(f"  Tokens  : {df.loc[i, 'tokens'][:80]}")
    print()

Applying normalization (may take ~1 min for 12k rows) ...
Done.

Sample tokens:
  ML text : betul sangat pasal calar dan kemek tu aku duduk atas kerusi letak atas peha seka
  Tokens  : pasal calar mek tu duduk kerusi letak peha jatuh mek cat gores pakai casing ok b

  ML text : teknologi foldable phone ni masih belum cukup matang lagi xberbaloi dgn harga yg
  Tokens  : teknologi foldable phone ni matang xberbaloi dgn harga yg kera nak mam skrin lev

  ML text : jadi android tpi version yg outdated atau lebih kpada android mid range yg tiada
  Tokens  : android tpi version yg outdated kpada android mid range yg tiada bypass charging

  ML text : minat gak pixel cuma harga agak mahal utk aku
  Tokens  : minat gak pixel harga mahal utk

  ML text : i nak tunggu xiaomi pro max
  Tokens  : nak tunggu xiaomi pro max



## 5. Sentiment Labeling

**Model:** [`cardiffnlp/twitter-xlm-roberta-base-sentiment`](https://huggingface.co/cardiffnlp/twitter-xlm-roberta-base-sentiment)

- Fine-tuned on 198M multilingual tweets across 8 languages including code-switched text.
- Handles Malay–English mixed comments well without any translation step.
- Labels: `negative` (0) · `neutral` (1) · `positive` (2)

**Input:** `clean_text` (light-cleaned) — emojis and casing preserved to match training distribution.  
**Output:** `sentiment_label` + `sentiment_score` (confidence of the predicted class).

> Batch size is set to **64 on GPU** and **16 on CPU** to balance speed and memory.

In [9]:
DEVICE     = 0 if torch.cuda.is_available() else -1
BATCH_SIZE = 64 if DEVICE == 0 else 16

print(f"Device     : {'GPU (cuda:0)' if DEVICE == 0 else 'CPU'}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Est. time  : {'~5 min' if DEVICE == 0 else '~45 min'} for 12k comments\n")

print("Loading cardiffnlp/twitter-xlm-roberta-base-sentiment ...")
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    device=DEVICE,
    truncation=True,
    max_length=128,
)
print("Model loaded.")

Device     : GPU (cuda:0)
Batch size : 64
Est. time  : ~5 min for 12k comments

Loading cardiffnlp/twitter-xlm-roberta-base-sentiment ...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 26772.15it/s]


Model loaded.


In [10]:
# Label map as safety net (model usually returns named labels directly)
LABEL_MAP = {
    "LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive",
    "negative": "negative", "neutral": "neutral",  "positive": "positive",
}

def run_sentiment(texts, pipe, batch_size):
    labels, scores = [], []
    for i in tqdm(range(0, len(texts), batch_size), desc="Classifying"):
        batch   = texts[i : i + batch_size]
        results = pipe(batch)
        for r in results:
            labels.append(LABEL_MAP.get(r["label"], r["label"]))
            scores.append(round(r["score"], 4))
    return labels, scores


# Use clean_text for inference; fall back to comment_text if empty
inference_texts = [
    c if c.strip() else o
    for c, o in zip(df["clean_text"].tolist(), df["comment_text"].tolist())
]

labels, scores = run_sentiment(inference_texts, sentiment_pipe, BATCH_SIZE)

df["sentiment_label"] = labels
df["sentiment_score"]  = scores

print("\nInference complete.")
print(df["sentiment_label"].value_counts())

Classifying: 100%|██████████| 188/188 [02:16<00:00,  1.38it/s]


Inference complete.
sentiment_label
neutral     5130
negative    3670
positive    3199
Name: count, dtype: int64


In [11]:
dist = df["sentiment_label"].value_counts()
pct  = df["sentiment_label"].value_counts(normalize=True) * 100

summary = pd.DataFrame({"count": dist, "percent": pct.round(1)})
print("=== Label Distribution ===")
print(summary.to_string())
print(f"\nTotal rows : {len(df):,}")

# Confidence score stats per label
print("\n=== Mean Confidence per Label ===")
print(df.groupby("sentiment_label")["sentiment_score"].mean().round(4))

# Flag low-confidence predictions (< 0.6) for awareness
low_conf = (df["sentiment_score"] < 0.6).sum()
print(f"\nLow-confidence predictions (<0.6) : {low_conf} ({low_conf/len(df)*100:.1f}%)")

=== Label Distribution ===
                 count  percent
sentiment_label                
neutral           5130     42.8
negative          3670     30.6
positive          3199     26.7

Total rows : 11,999

=== Mean Confidence per Label ===
sentiment_label
negative    0.6255
neutral     0.5549
positive    0.5989
Name: sentiment_score, dtype: float64

Low-confidence predictions (<0.6) : 6910 (57.6%)


## 6. Manual Verification Sample

A stratified random sample of **300 comments** (100 per class) is exported to `data/sample_manual_check.csv`.

Open the file in Excel or Google Sheets and add a `manual_label` column with your own judgment (`positive` / `neutral` / `negative`). Compare it against `sentiment_label` to estimate real-world labeling accuracy before training.

In [12]:
SAMPLE_PATH = r"c:\UTM things\Year 3\SEM 2\HDPD\Project 2\data\samples_check.csv"
N_PER_CLASS = 100

sample_frames = []
for label in ["positive", "neutral", "negative"]:
    subset = df[df["sentiment_label"] == label]
    n      = min(N_PER_CLASS, len(subset))
    sample_frames.append(subset.sample(n, random_state=42))

df_sample = pd.concat(sample_frames).sample(frac=1, random_state=42).reset_index(drop=True)

# Include the most useful columns for manual review
df_sample[[
    "comment_id", "video_title", "comment_text",
    "clean_text", "sentiment_label", "sentiment_score", "published_at"
]].to_csv(SAMPLE_PATH, index=False, encoding="utf-8-sig")

print(f"Sample saved  → {os.path.abspath(SAMPLE_PATH)}")
print(f"Rows          : {len(df_sample)}")
print(df_sample["sentiment_label"].value_counts().to_string())

Sample saved  → c:\UTM things\Year 3\SEM 2\HDPD\Project 2\data\samples_check.csv
Rows          : 300
sentiment_label
negative    100
neutral     100
positive    100


## 7. Save Cleaned Dataset

Final `data/cleaned_data.csv` contains the columns needed for all downstream tasks:

| Column | Used by |
|---|---|
| `comment_id` | Deduplication reference |
| `video_title` | Context / filtering |
| `comment_text` | Display / audit |
| `clean_text` | Transformer model training |
| `clean_text_ml` | TF-IDF feature input |
| `tokens` | TF-IDF feature input (pre-tokenized) |
| `sentiment_label` | Training target |
| `sentiment_score` | Confidence filter |
| `like_count` | Optional weighting |
| `published_at` | Time-series analysis |

In [13]:
OUTPUT_COLS = [
    "comment_id", "video_id", "video_title", "video_published_at",
    "comment_text", "clean_text", "clean_text_ml", "tokens",
    "sentiment_label", "sentiment_score",
    "like_count", "reply_count", "published_at",
]

OUT_PATH = r"c:\UTM things\Year 3\SEM 2\HDPD\Project 2\data\cleaned_data.csv"

df[OUTPUT_COLS].to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"Saved  → {os.path.abspath(OUT_PATH)}")
print(f"Shape  : {len(df):,} rows × {len(OUTPUT_COLS)} columns")
print(f"Size   : {os.path.getsize(OUT_PATH) / 1024:.1f} KB")

print("\n=== Final Label Distribution ===")
dist = df["sentiment_label"].value_counts()
pct  = df["sentiment_label"].value_counts(normalize=True) * 100
for label in ["positive", "neutral", "negative"]:
    print(f"  {label:<10} {dist[label]:>5,}  ({pct[label]:.1f}%)")

print("\n=== Class Balance Check ===")
majority = dist.max()
minority = dist.min()
ratio    = majority / minority
print(f"  Majority / Minority ratio : {ratio:.2f}x")
if ratio > 3:
    print("  ⚠  Imbalanced — consider oversampling (SMOTE) or class weights during training.")
else:
    print("  ✓  Reasonably balanced.")

Saved  → c:\UTM things\Year 3\SEM 2\HDPD\Project 2\data\cleaned_data.csv
Shape  : 11,999 rows × 13 columns
Size   : 5539.8 KB

=== Final Label Distribution ===
  positive   3,199  (26.7%)
  neutral    5,130  (42.8%)
  negative   3,670  (30.6%)

=== Class Balance Check ===
  Majority / Minority ratio : 1.60x
  ✓  Reasonably balanced.
